In [66]:
!pip install polars

In [67]:
import json
import subprocess
import re
import polars as pl

In [68]:
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)

In [69]:
# Get workspace info
workspace = wb("workspace", "describe")
GOOGLE_CLOUD_PROJECT = workspace["googleProjectId"]

# CYP2C_Cluster

In [71]:
!gcloud storage cp \
  --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/2c_cluster_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/2c_cluster_cohort_results.tsv to file://./2c_cluster_cohort_results.tsv
  Completed files 1/1 | 42.7MiB/42.7MiB                                        

Average throughput: 89.7MiB/s


In [6]:
# Read the TSV file into a Polars DataFrame
CYP2C_df = pl.read_csv("2c_cluster_cohort_results.tsv", separator="\t")

In [7]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(grouped_counts)

shape: (3, 4)
┌────────────┬─────────────────────────────────┬─────────────────┬────────┐
│ gene       ┆ genotype                        ┆ phenotype       ┆ len    │
│ ---        ┆ ---                             ┆ ---             ┆ ---    │
│ str        ┆ str                             ┆ str             ┆ u32    │
╞════════════╪═════════════════════════════════╪═════════════════╪════════╡
│ 2C_CLUSTER ┆ rs12777823 reference (G)/rs127… ┆ Variant Absent  ┆ 369248 │
│ 2C_CLUSTER ┆ rs12777823 reference (G)/rs127… ┆ Variant Present ┆ 149070 │
│ 2C_CLUSTER ┆ rs12777823 variant (A)/rs12777… ┆ Variant Present ┆ 17314  │
└────────────┴─────────────────────────────────┴─────────────────┴────────┘


# ABCG2

In [8]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
  --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/abcg2_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/abcg2_cohort_results.tsv to file://./abcg2_cohort_results.tsv
  Completed files 1/1 | 39.9MiB/39.9MiB                                        

Average throughput: 120.4MiB/s


In [9]:
# Read the TSV file into a Polars DataFrame
ABCG2_df = pl.read_csv("abcg2_cohort_results.tsv", separator="\t")

In [10]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = ABCG2_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(grouped_counts)

shape: (3, 4)
┌───────┬─────────────────────────────────┬────────────────────┬────────┐
│ gene  ┆ genotype                        ┆ phenotype          ┆ len    │
│ ---   ┆ ---                             ┆ ---                ┆ ---    │
│ str   ┆ str                             ┆ str                ┆ u32    │
╞═══════╪═════════════════════════════════╪════════════════════╪════════╡
│ ABCG2 ┆ rs2231142 variant (T)/rs223114… ┆ Poor Function      ┆ 8500   │
│ ABCG2 ┆ rs2231142 reference (G)/rs2231… ┆ Decreased Function ┆ 100088 │
│ ABCG2 ┆ rs2231142 reference (G)/rs2231… ┆ Normal Function    ┆ 427044 │
└───────┴─────────────────────────────────┴────────────────────┴────────┘


# CACNA1S

In [22]:
#Retrieve Pharmacogenomic table from All of us!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/pgx/high_concordance/cacna1s_cohort_results.tsv .
!gcloud storage cp \
  --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cacna1s_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cacna1s_cohort_results.tsv to file://./cacna1s_cohort_results.tsv
  Completed files 1/1 | 31.2MiB/31.2MiB                                        

Average throughput: 64.6MiB/s


In [23]:
CACNA1S_df = pl.read_csv(
    "cacna1s_cohort_results.tsv",
    separator="\t",
    null_values=["n/a"]
)

In [24]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CACNA1S_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1) # Don't print counts <20

polars.config.Config

In [26]:
phenotype_counts = (
    CACNA1S_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (2, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Uncertain Susceptibility        ┆ 535597 │
│ Malignant Hyperthermia Suscept… ┆ 35     │
└─────────────────────────────────┴────────┘


# CFTR

In [17]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
  --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cftr_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cftr_cohort_results.tsv to file://./cftr_cohort_results.tsv
  Completed files 1/1 | 27.1MiB/27.1MiB                                        

Average throughput: 109.2MiB/s


In [18]:
# Read the TSV file into a Polars DataFrame
CFTR_df = pl.read_csv("cftr_cohort_results.tsv", separator="\t")

In [19]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CFTR_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [20]:
phenotype_counts = (
    CFTR_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (3, 2)
┌────────────────────┬────────┐
│ phenotype          ┆ count  │
│ ---                ┆ ---    │
│ str                ┆ u32    │
╞════════════════════╪════════╡
│ normal_function    ┆ 526608 │
│ unknown_function   ┆ 5554   │
│ decreased_function ┆ 3470   │
└────────────────────┴────────┘


# CYP2D6

In [27]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp --billing-project=$GOOGLE_CLOUD_PROJECT gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp2d6_cohort_foxtrot_results.tsv .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp2d6_cohort_foxtrot_results.tsv to file://./cyp2d6_cohort_foxtrot_results.tsv
  Completed files 1/1 | 44.1MiB/44.1MiB                                        

Average throughput: 93.2MiB/s


In [28]:
# Read the TSV file into a Polars DataFrame
CYP2D6_df = pl.read_csv("cyp2d6_cohort_foxtrot_results.tsv", separator="\t")

In [29]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2D6_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [30]:
phenotype_counts = (
    CYP2D6_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ CYP2D6 Normal Metabolizer       ┆ 279865 │
│ CYP2D6 Intermediate Metabolize… ┆ 187682 │
│ CYP2D6 Indeterminate Metaboliz… ┆ 27802  │
│ CYP2D6 Poor Metabolizer         ┆ 26002  │
│ CYP2D6 Ultrarapid Metabolizer   ┆ 14305  │
└─────────────────────────────────┴────────┘


# G6PD

In [72]:
!gcloud storage cp \
  --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/g6pd_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/g6pd_cohort_results.tsv to file://./g6pd_cohort_results.tsv
  Completed files 1/1 | 21.5MiB/21.5MiB                                        

Average throughput: 122.4MiB/s


In [73]:
# Read the TSV file into a Polars DataFrame
G6PD_df = pl.read_csv("g6pd_cohort_results.tsv", separator="\t")

In [74]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = G6PD_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [75]:
phenotype_counts = (
    G6PD_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (4, 2)
┌───────────────┬────────┐
│ phenotype     ┆ count  │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
│ Normal        ┆ 510345 │
│ Variable      ┆ 23680  │
│ Deficient     ┆ 1070   │
│ Indeterminate ┆ 537    │
└───────────────┴────────┘


# NUDT15

In [76]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
  --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/nudt15_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/nudt15_cohort_results.tsv to file://./nudt15_cohort_results.tsv
  Completed files 1/1 | 20.5MiB/20.5MiB                                        

Average throughput: 113.0MiB/s


In [77]:
# Read the TSV file into a Polars DataFrame
NUDT15_df = pl.read_csv("nudt15_cohort_results.tsv", separator="\t")

In [78]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = NUDT15_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [79]:
phenotype_counts = (
    NUDT15_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 514558 │
│ Intermediate Metabolizer        ┆ 14521  │
│ Indeterminate                   ┆ 5943   │
│ Poor Metabolizer                ┆ 432    │
│ Possible Intermediate Metaboli… ┆ 178    │
└─────────────────────────────────┴────────┘


# RYR1

In [80]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/ryr1_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/ryr1_cohort_results.tsv to file://./ryr1_cohort_results.tsv
  Completed files 1/1 | 29.6MiB/29.6MiB                                        

Average throughput: 137.5MiB/s


In [81]:
# Read the TSV file into a Polars DataFrame
RYR1_df = pl.read_csv("ryr1_cohort_results.tsv", separator="\t")

In [82]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = RYR1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [84]:
phenotype_counts = (
    RYR1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") >= 20)
    .sort("count", descending=True)
)

print(phenotype_counts) #Don't print counts <20

shape: (2, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Uncertain Susceptibility        ┆ 535210 │
│ Malignant Hyperthermia Suscept… ┆ 408    │
└─────────────────────────────────┴────────┘


# TPMT

In [85]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/tpmt_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/tpmt_cohort_results.tsv to file://./tpmt_cohort_results.tsv
  Completed files 1/1 | 19.7MiB/19.7MiB                                        

Average throughput: 121.7MiB/s


In [86]:
# Read the TSV file into a Polars DataFrame
TPMT_df = pl.read_csv("tpmt_cohort_results.tsv", separator="\t")

In [87]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = TPMT_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [88]:
phenotype_counts = (
    TPMT_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 475123 │
│ Intermediate Metabolizer        ┆ 47130  │
│ Indeterminate                   ┆ 11718  │
│ Poor Metabolizer                ┆ 1017   │
│ Possible Intermediate Metaboli… ┆ 644    │
└─────────────────────────────────┴────────┘


# VKORC1

In [89]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/vkorc1_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/vkorc1_cohort_results.tsv to file://./vkorc1_cohort_results.tsv
  Completed files 1/1 | 39.4MiB/39.4MiB                                        

Average throughput: 123.6MiB/s


In [90]:
# Read the TSV file into a Polars DataFrame
VKORC1_df = pl.read_csv("vkorc1_cohort_results.tsv", separator="\t")

In [91]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = VKORC1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20
print(grouped_counts)

shape: (3, 4)
┌────────┬─────────────────────────────────┬─────────────────┬────────┐
│ gene   ┆ genotype                        ┆ phenotype       ┆ len    │
│ ---    ┆ ---                             ┆ ---             ┆ ---    │
│ str    ┆ str                             ┆ str             ┆ u32    │
╞════════╪═════════════════════════════════╪═════════════════╪════════╡
│ VKORC1 ┆ rs9923231 reference (C)/rs9923… ┆ Variant Absent  ┆ 236932 │
│ VKORC1 ┆ rs9923231 reference (C)/rs9923… ┆ Variant Present ┆ 220426 │
│ VKORC1 ┆ rs9923231 variant (T)/rs992323… ┆ Variant Present ┆ 78272  │
└────────┴─────────────────────────────────┴─────────────────┴────────┘


In [92]:
phenotype_counts = (
    VKORC1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts) #Note that variant present includes both hom and het counts

shape: (2, 2)
┌─────────────────┬────────┐
│ phenotype       ┆ count  │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ Variant Present ┆ 298698 │
│ Variant Absent  ┆ 236932 │
└─────────────────┴────────┘


# CYP2B6

In [93]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp2b6_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp2b6_cohort_results.tsv to file://./cyp2b6_cohort_results.tsv
  Completed files 1/1 | 23.0MiB/23.0MiB                                        

Average throughput: 74.9MiB/s


In [94]:
# Read the TSV file into a Polars DataFrame
CYP2B6_df = pl.read_csv("cyp2b6_cohort_results.tsv", separator="\t")

In [95]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2B6_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [96]:
phenotype_counts = (
    CYP2B6_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (5, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ u32    │
╞══════════════════════════╪════════╡
│ normal_metabolizer       ┆ 437184 │
│ intermediate_metabolizer ┆ 61053  │
│ ultrarapid_metabolizer   ┆ 21657  │
│ unknown_metabolizer      ┆ 15196  │
│ poor_metabolizer         ┆ 542    │
└──────────────────────────┴────────┘


# CYP2C19

In [97]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/low_conf/cyp2c19_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/low_conf/cyp2c19_cohort_results.tsv to file://./cyp2c19_cohort_results.tsv
  Completed files 1/1 | 22.0MiB/22.0MiB                                        

Average throughput: 77.2MiB/s


In [98]:
# Read the TSV file into a Polars DataFrame
CYP2C19_df = pl.read_csv("cyp2c19_cohort_results.tsv", separator="\t")

In [99]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C19_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [100]:
phenotype_counts = (
    CYP2C19_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (8, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 217824 │
│ Intermediate Metabolizer        ┆ 142059 │
│ Rapid Metabolizer               ┆ 128163 │
│ Ultrarapid Metabolizer          ┆ 21707  │
│ Poor Metabolizer                ┆ 15261  │
│ Indeterminate                   ┆ 6802   │
│ Likely Intermediate Metabolize… ┆ 3052   │
│ Likely Poor Metabolizer         ┆ 764    │
└─────────────────────────────────┴────────┘


# CYP3A5

In [101]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp3a5_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp3a5_cohort_results.tsv to file://./cyp3a5_cohort_results.tsv
  Completed files 1/1 | 20.5MiB/20.5MiB                                        

Average throughput: 85.1MiB/s


In [102]:
# Read the TSV file into a Polars DataFrame
CYP3A5_df = pl.read_csv("cyp3a5_cohort_results.tsv", separator="\t")

In [103]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP3A5_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [126]:
phenotype_counts = (
    CYP3A5_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") >= 20) #Don't print counts <20
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (4, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ u32    │
╞══════════════════════════╪════════╡
│ Poor Metabolizer         ┆ 376957 │
│ Intermediate Metabolizer ┆ 129235 │
│ Normal Metabolizer       ┆ 29377  │
│ Indeterminate            ┆ 40     │
└──────────────────────────┴────────┘


# CYP4F2

In [105]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp4f2_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp4f2_cohort_results.tsv to file://./cyp4f2_cohort_results.tsv
  Completed files 1/1 | 22.8MiB/22.8MiB                                        

Average throughput: 114.2MiB/s


In [106]:
# Read the TSV file into a Polars DataFrame
CYP4F2_df = pl.read_csv("cyp4f2_cohort_results.tsv", separator="\t")

In [107]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP4F2_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(grouped_counts)

shape: (7, 4)
┌────────┬─────────────────────────────┬────────────────────────┬────────┐
│ gene   ┆ genotype                    ┆ phenotype              ┆ len    │
│ ---    ┆ ---                         ┆ ---                    ┆ ---    │
│ str    ┆ str                         ┆ str                    ┆ u32    │
╞════════╪═════════════════════════════╪════════════════════════╪════════╡
│ CYP4F2 ┆ *1/*1                       ┆ CYP4F2 Variant Absent  ┆ 273009 │
│ CYP4F2 ┆ *1/*3                       ┆ CYP4F2 Variant Present ┆ 83344  │
│ CYP4F2 ┆ *1/*2                       ┆ CYP4F2 Variant Absent  ┆ 31239  │
│ CYP4F2 ┆ *2/*2                       ┆ CYP4F2 Variant Absent  ┆ 3449   │
│ CYP4F2 ┆ *3/*3                       ┆ CYP4F2 Variant Present ┆ 7537   │
│ CYP4F2 ┆ Indeterminate/Indeterminate ┆ CYP4F2 Indeterminate   ┆ 31986  │
│ CYP4F2 ┆ *3/*2                       ┆ CYP4F2 Variant Present ┆ 105068 │
└────────┴─────────────────────────────┴────────────────────────┴────────┘


In [108]:
phenotype_counts = (
    CYP4F2_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (3, 2)
┌────────────────────────┬────────┐
│ phenotype              ┆ count  │
│ ---                    ┆ ---    │
│ str                    ┆ u32    │
╞════════════════════════╪════════╡
│ CYP4F2 Variant Absent  ┆ 307697 │
│ CYP4F2 Variant Present ┆ 195949 │
│ CYP4F2 Indeterminate   ┆ 31986  │
└────────────────────────┴────────┘


# DPYD

In [109]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/low_conf/dpyd_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/low_conf/dpyd_cohort_results.tsv to file://./dpyd_cohort_results.tsv
  Completed files 1/1 | 45.2MiB/45.2MiB                                        

Average throughput: 124.7MiB/s


In [110]:
# Read the TSV file into a Polars DataFrame
DPYD_df = pl.read_csv("dpyd_cohort_results.tsv", separator="\t")

In [111]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = DPYD_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [112]:
phenotype_counts = (
    DPYD_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (4, 2)
┌───────────────────────────────┬────────┐
│ phenotype                     ┆ count  │
│ ---                           ┆ ---    │
│ str                           ┆ u32    │
╞═══════════════════════════════╪════════╡
│ DPYD Normal Metabolizer       ┆ 507955 │
│ DPYD Intermediate Metabolizer ┆ 27162  │
│ Indeterminate                 ┆ 400    │
│ DPYD Poor Metabolizer         ┆ 113    │
└───────────────────────────────┴────────┘


# SLCO1B1

In [113]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/low_conf/slco1b1_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/low_conf/slco1b1_cohort_results.tsv to file://./slco1b1_cohort_results.tsv
  Completed files 1/1 | 20.4MiB/20.4MiB                                        

Average throughput: 64.2MiB/s


In [114]:
# Read the TSV file into a Polars DataFrame
SLCO1B1_df = pl.read_csv("slco1b1_cohort_results.tsv", separator="\t")

In [115]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = SLCO1B1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [116]:
phenotype_counts = (
    SLCO1B1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (6, 2)
┌─────────────────────────────┬────────┐
│ phenotype                   ┆ count  │
│ ---                         ┆ ---    │
│ str                         ┆ u32    │
╞═════════════════════════════╪════════╡
│ Normal Function             ┆ 359355 │
│ Decreased Function          ┆ 115342 │
│ Indeterminate               ┆ 31268  │
│ Increased Function          ┆ 17076  │
│ Poor Function               ┆ 10531  │
│ Possible Decreased Function ┆ 2059   │
└─────────────────────────────┴────────┘


# UGT1A1

In [117]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/ugt1a1_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/ugt1a1_cohort_results.tsv to file://./ugt1a1_cohort_results.tsv
  Completed files 1/1 | 23.5MiB/23.5MiB                                        

Average throughput: 114.0MiB/s


In [118]:
# Read the TSV file into a Polars DataFrame
UGT1A1_df = pl.read_csv("ugt1a1_cohort_results.tsv", separator="\t")

In [119]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = UGT1A1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [120]:
phenotype_counts = (
    UGT1A1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (4, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ u32    │
╞══════════════════════════╪════════╡
│ Intermediate Metabolizer ┆ 242578 │
│ Normal Metabolizer       ┆ 222365 │
│ Poor Metabolizer         ┆ 70307  │
│ Indeterminate            ┆ 382    │
└──────────────────────────┴────────┘


# CYP2C9

In [121]:
#Retrieve Pharmacogenomic table from All of us
!gcloud storage cp \
   --billing-project=$GOOGLE_CLOUD_PROJECT\
  gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp2c9_cohort_results.tsv \
  .

Copying gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx/high_conf/cyp2c9_cohort_results.tsv to file://./cyp2c9_cohort_results.tsv
  Completed files 1/1 | 23.4MiB/23.4MiB                                        

Average throughput: 118.0MiB/s


In [122]:
# Read the TSV file into a Polars DataFrame
CYP2C9_df = pl.read_csv("cyp2c9_cohort_results.tsv", separator="\t", null_values=["n/a", "NA", ""],)

In [123]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C9_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [124]:
phenotype_counts = (
    CYP2C9_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(phenotype_counts)

shape: (4, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ u32    │
╞══════════════════════════╪════════╡
│ Normal Metabolizer       ┆ 365867 │
│ Intermediate Metabolizer ┆ 159039 │
│ Poor Metabolizer         ┆ 9202   │
│ Indeterminate            ┆ 1524   │
└──────────────────────────┴────────┘
